# Phase 0 — Dataset Integration
**What:** combine the 12 original UCI station CSVs into one canonical dataset with a `datetime` column.  
**Why:** every later phase works from a single, consistently ordered table.  
**Input:** `PRSA_Data_20130301-20170228/` (original UCI files).  
**Output:** `data/raw/beijing_air_quality.csv` and `data/audit/phase0_integration_continuity.csv`.  
**Decision:** continue only if all 12 stations load with the expected schema and 420,768 rows.

> Reconstructed for this repository: the original Phase 0 notebook was not included in the submission.
> Its outputs were verified to be byte-identical to the team's original Phase 0 outputs.

## 1. Load the 12 station files

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_config import *

import pandas as pd

if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"{SOURCE_DIR} not found. Download the dataset from "
        "https://archive.ics.uci.edu/dataset/501 and extract it into the project root."
    )

frames = []
for station in EXPECTED_STATIONS:
    path = SOURCE_DIR / f"PRSA_Data_{station}_20130301-20170228.csv"
    part = pd.read_csv(path)
    assert part.columns.tolist() == SOURCE_COLUMNS, f"Unexpected columns in {path.name}"
    frames.append(part)
    print(f"{station:<14} {len(part):>6} rows")

## 2. Integrate and add `datetime`

In [ ]:
df = pd.concat(frames, ignore_index=True)
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)

assert df.columns.tolist() == CANONICAL_COLUMNS
assert len(df) == 420_768
print("Shape:", df.shape)
display(df.head())

## 3. Check hourly continuity and save

In [ ]:
rows = []
for station, g in df.groupby("station", sort=True):
    diff = g["datetime"].diff().dropna()
    rows.append({
        "station": station,
        "rows": len(g),
        "start": g["datetime"].min(),
        "end": g["datetime"].max(),
        "non_hourly_gaps": int((diff != pd.Timedelta(hours=1)).sum())
    })
continuity_df = pd.DataFrame(rows)
display(continuity_df)

RAW_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(CANONICAL_RAW, index=False)
continuity_df.to_csv(AUDIT_DIR / "phase0_integration_continuity.csv", index=False)
print("Saved:", CANONICAL_RAW)
print("PHASE 0: PASS")